# W2 Day4 (04/17 周四): [实战] nanoGPT

## 学习目标
- **理论 (1-1.5h)**: 完整 decoder-only 架构 + 训练循环 + 文本生成; KV Cache 原理与实现; 采样策略 (Greedy / Top-k / Top-p / Beam Search)
- **实践 (5h)**: 从零手写 nanoGPT (字符级) + 训练 + KV Cache 加速 + 多种采样策略实现
- **产出物**: nanoGPT (GitHub) — 可训练、可生成、含 KV Cache 与采样对比

## 核心问题 (面试高频)
1. Decoder-only 与 Encoder-Decoder 的区别？为什么 GPT 选 decoder-only？
2. Causal mask 的实现细节？为什么是下三角？
3. KV Cache 节省了什么？显存占用怎么算？
4. Top-k / Top-p / Temperature 的物理含义？什么场景用哪个？
5. KV Cache 在 batch 推理时的复杂度？(为后续 PagedAttention 铺路)

---

## 目录
1. [GPT 架构总览](#1)
2. [Causal Self-Attention](#2)
3. [Transformer Block (Pre-LN)](#3)
4. [完整 GPT 模型](#4)
5. [字符级数据准备](#5)
6. [训练循环](#6)
7. [生成: 朴素实现](#7)
8. [KV Cache 加速](#8)
9. [采样策略对比](#9)
10. [总结 & 思考题](#10)


---
## 1. GPT 架构总览 <a id='1'></a>

### 1.1 Decoder-only 设计

GPT (Generative Pre-trained Transformer) 是 **decoder-only** 架构，与原始 Transformer 的关键差异:

| 组件 | 原始 Transformer | GPT |
|---|---|---|
| Encoder | ✅ | ❌ |
| Decoder | ✅ (含 cross-attention) | ✅ (无 cross-attention) |
| Attention | Encoder 双向 + Decoder masked | **全部 masked (causal)** |
| 训练任务 | seq2seq | **next token prediction** |

### 1.2 为什么 decoder-only 赢了？

- **训练简单**: 只需要 next-token prediction，不需要构造 (input, target) 对
- **scaling 友好**: 同样的参数量，decoder-only 能用更多数据
- **任务统一**: 翻译、摘要、QA 都可以表达为「续写」
- **In-context learning 涌现**: 长上下文 + causal mask 让模型学会从上下文学习

### 1.3 整体数据流

```
input_ids (B, T)
    ↓
token_embed + pos_embed → (B, T, C)
    ↓
[Block × N]
    └─ LN → MHSA (causal) → +residual
    └─ LN → FFN (4×) → +residual
    ↓
LN → linear → (B, T, vocab_size)
    ↓
loss = CE(logits[:, :-1], targets[:, 1:])
```

### 1.4 关键超参数 (与 GPT-2 对齐)

| 参数 | 含义 | nanoGPT (本课) | GPT-2 small | GPT-3 |
|---|---|---|---|---|
| n_layer | Transformer 层数 | 4 | 12 | 96 |
| n_head | 注意力头数 | 4 | 12 | 96 |
| n_embd | 隐藏维度 C | 128 | 768 | 12288 |
| block_size | 最大上下文 T | 64 | 1024 | 2048 |
| vocab_size | 词表大小 | ~65 (字符级) | 50257 (BPE) | 50257 |


In [ ]:
import math
import time
import os
import urllib.request
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(1337)
np.random.seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print("=" * 60)


@dataclass
class GPTConfig:
    block_size: int = 64       # T: 最大上下文
    vocab_size: int = 65       # V: 字符级词表 (Shakespeare 大约 65)
    n_layer: int = 4           # N: Transformer 层数
    n_head: int = 4            # H: 头数
    n_embd: int = 128          # C: 隐藏维度
    dropout: float = 0.1
    bias: bool = False         # GPT-2 风格: 关闭 LayerNorm/Linear 的 bias


cfg = GPTConfig()
print(f"\n配置: layer={cfg.n_layer}, head={cfg.n_head}, embd={cfg.n_embd}, "
      f"block={cfg.block_size}")
print(f"每个 head 维度: {cfg.n_embd // cfg.n_head}")


---
## 2. Causal Self-Attention <a id='2'></a>

### 2.1 Causal Mask: 为什么是下三角？

GPT 训练目标是「**给定前 t 个 token，预测第 t+1 个**」，所以位置 t 的 query 只能看到位置 ≤ t 的 key/value。

```
mask:           允许看的位置 (i ≥ j 才能看)
[1, 0, 0, 0]    位置0: 只能看位置0
[1, 1, 0, 0]    位置1: 看位置0, 1
[1, 1, 1, 0]    位置2: 看位置0, 1, 2
[1, 1, 1, 1]    位置3: 看位置0, 1, 2, 3
```

被 mask 掉的位置在 softmax **之前**赋值为 `-inf`，softmax 后就是 0，不贡献信息。

### 2.2 实现要点

- **合并 QKV 投影**: 一个 `Linear(C, 3C)` 一次算出 Q, K, V (节省一次矩阵乘法的开销)
- **多头分裂**: `(B, T, 3C)` → reshape 成 `(B, H, T, C/H)`
- **缩放**: 除以 `sqrt(d_k)` 防止 softmax 进入饱和区
- **register_buffer**: mask 不是参数但要随模型搬到 GPU

### 2.3 Flash Attention

PyTorch 2.0+ 的 `F.scaled_dot_product_attention` 自动用 Flash Attention，比手写快 2-4×、显存省一半。生产代码应该用它，但**手写一次** + 知道它做了什么是面试硬要求。


In [ ]:
class CausalSelfAttention(nn.Module):
    """
    手写 Causal Multi-Head Self-Attention
    
    输入:  x (B, T, C)
    输出:  y (B, T, C)
    
    数学: y = softmax((Q K^T / sqrt(d_k)) + mask) V
    """
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0, "n_embd 必须能被 n_head 整除"
        
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        
        # QKV 合并投影 (一个矩阵乘法搞定)
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        # 输出投影
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        
        # Causal mask (下三角)，注册为 buffer (不是参数，但跟模型走)
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer('mask', mask.view(1, 1, cfg.block_size, cfg.block_size))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # (B, T, C) → (B, T, 3C) → 拆成 Q, K, V
        qkv = self.qkv(x)
        q, k, v = qkv.split(self.n_embd, dim=2)  # 每个 (B, T, C)
        
        # 多头分裂: (B, T, C) → (B, H, T, C/H)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # 注意力分数: (B, H, T, T)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal mask: 用 -inf 掩盖未来位置
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        
        # softmax 归一化 → 概率分布
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        
        # 加权求和 V: (B, H, T, T) @ (B, H, T, C/H) → (B, H, T, C/H)
        y = att @ v
        
        # 合并多头: (B, H, T, C/H) → (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # 输出投影 + dropout
        y = self.resid_dropout(self.proj(y))
        return y


# 验证
attn = CausalSelfAttention(cfg).to(device)
x = torch.randn(2, 16, cfg.n_embd).to(device)
y = attn(x)
print(f"输入: {tuple(x.shape)}  →  输出: {tuple(y.shape)}")
print(f"参数量: {sum(p.numel() for p in attn.parameters()):,}")

# 验证 causal: 第 t 个位置不应该被未来位置影响
# 通过修改未来位置，看当前位置输出是否变化
x2 = x.clone()
x2[:, 8:, :] = torch.randn_like(x2[:, 8:, :])  # 改变 t=8 之后的位置
y2 = attn(x2)
diff_before_8 = (y[:, :8] - y2[:, :8]).abs().max().item()
diff_after_8 = (y[:, 8:] - y2[:, 8:]).abs().max().item()
print(f"\nCausal 验证:")
print(f"  改变 t=8 之后输入, t<8 的输出差异: {diff_before_8:.2e}  ← 应≈0 (causal)")
print(f"  改变 t=8 之后输入, t≥8 的输出差异: {diff_after_8:.2e}  ← 应>0")


---
## 3. Transformer Block (Pre-LN) <a id='3'></a>

### 3.1 Pre-LN vs Post-LN

```
Post-LN (原始 Transformer):       Pre-LN (GPT-2/LLaMA):
x → MHSA → +x → LN                x → LN → MHSA → +x
  → FFN  → +x → LN                  → LN → FFN  → +x
```

**为什么 GPT-2 之后都改用 Pre-LN？**
- Post-LN 在层数深时梯度不稳定 (需要 warmup)
- Pre-LN 残差路径上没有 LN，梯度可以直接传到底层 → 更易训练
- 代价: 表达能力略弱，最后需要再加一个 final LN

### 3.2 FFN: 4× 隐藏维度

GPT-2 的 FFN 中间层维度是 `4 * n_embd`。这个 4× 来自实验观察 (没有理论必然性)，LLaMA 改成了 SwiGLU + ~2.67×。

### 3.3 GELU vs ReLU

GPT-2 用 GELU (`x * Φ(x)`)，比 ReLU 平滑 → 训练稳定性更好。


In [ ]:
class MLP(nn.Module):
    """FFN: 两层 MLP，中间 4× 扩展"""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
    
    def forward(self, x):
        x = self.fc(x)
        x = self.gelu(x)
        x = self.proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    """
    Transformer Block (Pre-LN 风格)
    
    x → LN → Attn → +x → LN → MLP → +x
    """
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # 注意: LN 先于 attn (Pre-LN)
        x = x + self.mlp(self.ln2(x))
        return x


# 验证
block = Block(cfg).to(device)
x = torch.randn(2, 16, cfg.n_embd).to(device)
y = block(x)
print(f"Block 输入: {tuple(x.shape)}  →  输出: {tuple(y.shape)}")
print(f"Block 参数量: {sum(p.numel() for p in block.parameters()):,}")
print(f"  其中 Attn: {sum(p.numel() for p in block.attn.parameters()):,}")
print(f"  其中 MLP:  {sum(p.numel() for p in block.mlp.parameters()):,}")
print(f"  其中 LN:   {sum(p.numel() for p in block.ln1.parameters()) + sum(p.numel() for p in block.ln2.parameters()):,}")
print(f"\n💡 MLP 占了大部分参数 (4× 扩展) — 这也是为什么 MoE 主要稀疏化 MLP")


---
## 4. 完整 GPT 模型 <a id='4'></a>

### 4.1 组装清单
1. Token embedding: `(V, C)`
2. Position embedding: `(T_max, C)` — GPT-2 用学习的位置嵌入；LLaMA 用 RoPE (W3D4 会讲)
3. N 个 Block
4. Final LayerNorm
5. LM Head: `(C, V)` — 注意可以与 token embedding **权重共享** (weight tying)，省 V×C 参数

### 4.2 Weight Tying

`lm_head.weight = token_embed.weight` — 嵌入和输出投影共享权重。

- **节省参数**: V×C (字符级 65×128 ≈ 0.008M; GPT-2 50257×768 ≈ 39M)
- **效果略好或持平**: 论文 *Using the Output Embedding to Improve Language Models* 验证

### 4.3 参数量公式

总参数 ≈ `V·C (embedding) + N·(12·C² + ε) (blocks) + V·C (lm_head, 若不 tie)`

每个 Block ≈ 12C² (4C² QKV 投影 + 4C² 输出投影 + 8C² FFN 两个 linear)


In [ ]:
class GPT(nn.Module):
    """完整的 GPT (decoder-only)"""
    
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_embed = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.dropout = nn.Dropout(cfg.dropout)
        
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        
        # Weight tying: 节省 V×C 参数
        self.lm_head.weight = self.token_embed.weight
        
        # 初始化 (GPT-2 风格)
        self.apply(self._init_weights)
        # Residual 投影特殊缩放: 防止深层方差爆炸
        for pn, p in self.named_parameters():
            if pn.endswith('proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        """
        idx:     (B, T) — 输入 token ids
        targets: (B, T) — 移位 target (用于训练)
        
        返回:
        - logits: (B, T, V)
        - loss:   标量 (如果 targets 不为 None)
        """
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"序列长度 {T} > block_size {self.cfg.block_size}"
        
        pos = torch.arange(T, device=idx.device)               # (T,)
        tok_emb = self.token_embed(idx)                        # (B, T, C)
        pos_emb = self.pos_embed(pos)                          # (T, C)  → 广播到 (B, T, C)
        x = self.dropout(tok_emb + pos_emb)
        
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        
        logits = self.lm_head(x)                               # (B, T, V)
        
        loss = None
        if targets is not None:
            # CrossEntropy 期望 (B*T, V), (B*T,)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1
            )
        return logits, loss
    
    def num_params(self, non_embedding=True):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            # 减去 pos_embed (因为是参数但不算 "模型容量")
            n -= self.pos_embed.weight.numel()
        return n


model = GPT(cfg).to(device)
print(f"GPT 模型参数量: {model.num_params():,}  ({model.num_params()/1e6:.2f}M)")
print(f"  token_embed:  {model.token_embed.weight.numel():,}  (与 lm_head 共享)")
print(f"  pos_embed:    {model.pos_embed.weight.numel():,}")
print(f"  {cfg.n_layer} blocks: {sum(p.numel() for p in model.blocks.parameters()):,}")

# 前向 sanity check
x = torch.randint(0, cfg.vocab_size, (2, 16)).to(device)
y = torch.randint(0, cfg.vocab_size, (2, 16)).to(device)
logits, loss = model(x, y)
print(f"\n前向: idx{tuple(x.shape)} → logits{tuple(logits.shape)}, loss={loss.item():.4f}")
print(f"初始 loss ≈ ln(vocab_size) = {math.log(cfg.vocab_size):.4f} ← 随机预测的理论值")


---
## 5. 字符级数据准备 <a id='5'></a>

我们用 **Tiny Shakespeare** (1MB 莎士比亚剧本) 做字符级训练:
- 词表: 文本中所有出现的字符 (~65 个)
- 切分: 90% train / 10% val
- batch 采样: 随机起点切 `block_size+1` 长的片段

### 字符级 vs BPE
- 字符级: 词表小、不会 OOV、训练慢 (序列长)
- BPE: 词表大、压缩好、训练快 (主流 LLM 选择)

教学场景用字符级更直观。


In [ ]:
# 下载 Tiny Shakespeare (~1MB)
data_dir = '/tmp/nanogpt_data'
os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, 'shakespeare.txt')

if not os.path.exists(data_path):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    try:
        urllib.request.urlretrieve(url, data_path)
        print(f"下载完成: {data_path}")
    except Exception as e:
        # 离线 fallback: 用一段重复文本作为演示数据
        print(f"下载失败 ({e}), 使用 fallback 文本")
        sample = ("To be, or not to be, that is the question:\n"
                  "Whether 'tis nobler in the mind to suffer\n"
                  "The slings and arrows of outrageous fortune,\n"
                  "Or to take arms against a sea of troubles\n"
                  "And by opposing end them.\n\n") * 500
        with open(data_path, 'w') as f:
            f.write(sample)
else:
    print(f"已存在: {data_path}")

with open(data_path, 'r') as f:
    text = f.read()

print(f"\n数据集大小: {len(text):,} 字符")
print(f"前 200 字符:\n{text[:200]}")


In [ ]:
# 构建字符级 tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"词表大小: {vocab_size}")
print(f"前 30 个字符: {''.join(chars[:30])!r}")
print(f"\n编码 'Hello': {encode('Hello') if all(c in stoi for c in 'Hello') else '(部分字符不在词表中)'}")
test_str = chars[0] + chars[1] + chars[2]
print(f"编码/解码 round-trip 测试: {test_str!r} → {encode(test_str)} → {decode(encode(test_str))!r}")

# 更新 config 以匹配实际 vocab
cfg = GPTConfig(vocab_size=vocab_size)
print(f"\n更新后的 vocab_size: {cfg.vocab_size}")

# 切分 train/val
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"\nTrain: {len(train_data):,} tokens")
print(f"Val:   {len(val_data):,} tokens")


In [ ]:
def get_batch(split, batch_size=32, block_size=cfg.block_size):
    """
    从数据中随机采样一个 batch
    
    每个样本: (x, y), 其中 y 是 x 向后移 1 位
    例如: x = 'hello world', y = 'ello world!'
    """
    data = train_data if split == 'train' else val_data
    # 随机起点
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)


# 演示一个 batch
xb, yb = get_batch('train', batch_size=2, block_size=16)
print(f"x shape: {tuple(xb.shape)},  y shape: {tuple(yb.shape)}")
print(f"\n样本 0:")
print(f"  x = {decode(xb[0].tolist())!r}")
print(f"  y = {decode(yb[0].tolist())!r}")
print(f"\n💡 y[i] = x[i+1] (向后移 1 位) — 这就是 next-token prediction")


---
## 6. 训练循环 <a id='6'></a>

### 训练设置 (教学规模)
- 优化器: AdamW (`lr=3e-4`, `betas=(0.9, 0.95)`, `weight_decay=0.1`)
- 学习率: 余弦退火 + warmup
- batch_size: 32
- iterations: 2000 (CPU ~5 min, GPU ~30 sec)

GPT-2/3 用的是更复杂的方案，但核心一样: AdamW + cosine + warmup。


In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=20):
    """在 train/val 上各跑 eval_iters 个 batch，取平均 loss"""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split, batch_size=32)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


# 重新创建模型 (vocab_size 更新了)
model = GPT(cfg).to(device)
print(f"训练模型参数量: {model.num_params()/1e6:.2f}M")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1,
)

# 训练
max_iters = 2000
eval_interval = 200
batch_size = 32

train_losses_log = []
val_losses_log = []
log_iters = []

print(f"\n开始训练 ({max_iters} iters, batch_size={batch_size})...")
t0 = time.time()

for iter_num in range(max_iters):
    # 周期性评估
    if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
        losses = estimate_loss(model)
        train_losses_log.append(losses['train'])
        val_losses_log.append(losses['val'])
        log_iters.append(iter_num)
        elapsed = time.time() - t0
        print(f"  iter {iter_num:5d} | train {losses['train']:.4f} | val {losses['val']:.4f} | "
              f"{elapsed:.1f}s")
    
    # 训练一步
    xb, yb = get_batch('train', batch_size=batch_size)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

total_time = time.time() - t0
print(f"\n训练完成: {total_time:.1f}s ({max_iters/total_time:.1f} it/s)")


In [ ]:
# 训练曲线
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(log_iters, train_losses_log, 'o-', label='train', color='#1976D2')
ax.plot(log_iters, val_losses_log, 's-', label='val', color='#E64A19')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title(f'nanoGPT 训练曲线 ({model.num_params()/1e6:.2f}M params)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(math.log(cfg.vocab_size), color='gray', linestyle='--', alpha=0.5,
           label=f'random (ln V = {math.log(cfg.vocab_size):.2f})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n💡 观察: train loss 应该稳定下降，val loss 跟随但略高")
print(f"💡 字符级 Shakespeare 上 ~1.5-1.8 是常见水平 (per char)")


---
## 7. 生成: 朴素实现 <a id='7'></a>

### 7.1 自回归生成

```
prompt: 'ROME'
step 1: 输入 'ROME', 预测 'O' → 'ROMEO'
step 2: 输入 'ROMEO', 预测 ':' → 'ROMEO:'
step 3: 输入 'ROMEO:', 预测 ' ' → 'ROMEO: '
...
```

### 7.2 朴素实现的复杂度

每步生成都把**整个上下文**重新过一遍模型:
- 第 1 步: 处理长度 4 的序列
- 第 2 步: 处理长度 5 (前 4 个其实已经算过了！)
- 第 t 步: 处理长度 4+t

总复杂度 **O(T²)**，浪费严重。这就是 KV Cache 要解决的问题。


In [ ]:
@torch.no_grad()
def generate_naive(model, idx, max_new_tokens, temperature=1.0):
    """
    朴素生成: 每步都重新过整个上下文
    
    idx: (B, T_init) 起始 token ids
    """
    model.eval()
    for _ in range(max_new_tokens):
        # 截断到 block_size (位置嵌入的限制)
        idx_cond = idx if idx.size(1) <= cfg.block_size else idx[:, -cfg.block_size:]
        
        # 前向: 注意 logits 的形状是 (B, T, V)
        logits, _ = model(idx_cond)
        
        # 只取最后一个时间步的 logits
        logits = logits[:, -1, :] / temperature   # (B, V)
        
        # softmax → 采样
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)   # (B, 1)
        
        # 拼接到序列
        idx = torch.cat((idx, idx_next), dim=1)
    
    model.train()
    return idx


# 测试生成
prompt = "ROMEO:"
context = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
print(f"Prompt: {prompt!r}\n")

t0 = time.time()
out = generate_naive(model, context, max_new_tokens=200, temperature=1.0)
elapsed = time.time() - t0

generated = decode(out[0].tolist())
print(f"生成 ({elapsed:.2f}s, {200/elapsed:.0f} tok/s):")
print("-" * 60)
print(generated)
print("-" * 60)
print(f"\n💡 字符级模型 + 2000 iter 训练 → 能学到莎士比亚的 '形' 但不太懂语义")


---
## 8. KV Cache 加速 <a id='8'></a>

### 8.1 核心洞察

自回归生成中，**前面位置的 K, V 不会变化**。

```
step t   : 输入 [x_1, x_2, ..., x_t]
            算出 K_1..K_t, V_1..V_t
step t+1 : 输入 [x_1, x_2, ..., x_t, x_{t+1}]
            K_1..K_t 和 V_1..V_t 跟上一步完全一样！
            只需要新算 K_{t+1}, V_{t+1}
```

**KV Cache**: 把每层的 K, V 缓存起来，每步只增量计算新 token 的 K, V。

### 8.2 复杂度变化

| | 朴素 | KV Cache |
|---|---|---|
| 每步 attention | O(t·d) | O(t·d) (Q 长度 1，K/V 长度 t) |
| 每步前向 | O(t² · d · L) | **O(t · d · L)** |
| 总生成 (n tokens) | O(n³ · d) | **O(n² · d)** |
| 显存增量 | 0 | O(n · d · L · 2) (K + V) |

### 8.3 显存占用

GPT-2 (n_layer=12, n_embd=768) 每个 token 的 KV Cache:
- `2 (K+V) × 12 (layer) × 768 (C) × 2 (fp16) = 36864 bytes ≈ 36 KB / token`
- 1024 上下文: **36 MB**, batch=32 → **1.1 GB**
- LLaMA-7B (32 layer, 4096 dim, GQA 4 倍) 每 token 仍要 64 KB → 这就是为什么需要 PagedAttention (W7 会讲)

### 8.4 实现要点
- 改造 attention 接受可选的 `kv_cache` 参数
- 推理时 Q 用单个 token 算，K/V 拼接历史 + 当前
- causal mask 不再需要 (因为 query 只有 1 个)


In [ ]:
class CausalSelfAttentionKV(nn.Module):
    """支持 KV Cache 的 Causal Self-Attention"""
    
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer('mask', mask.view(1, 1, cfg.block_size, cfg.block_size))
    
    def forward(self, x, kv_cache=None):
        """
        x: (B, T, C) — T=1 在生成时 (除了第一次 prefill)
        kv_cache: (k_cache, v_cache) 或 None
                  k_cache: (B, H, T_past, head_dim)
        
        返回:
        - y:        (B, T, C)
        - new_cache: (k, v) 更新后的 cache
        """
        B, T, C = x.shape
        
        qkv = self.qkv(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # 拼接 KV cache
        if kv_cache is not None:
            k_past, v_past = kv_cache
            k = torch.cat([k_past, k], dim=2)
            v = torch.cat([v_past, v], dim=2)
        
        new_cache = (k, v)
        T_full = k.size(2)
        T_q = q.size(2)
        
        # Attention: 只对 q 这 T_q 个位置算分数
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # 当 T_q == 1 (单步生成) 不需要 mask；prefill 时需要
        if T_q > 1:
            # 取 mask 的右下角 T_q × T_full
            att = att.masked_fill(self.mask[:, :, T_full-T_q:T_full, :T_full] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        
        y = att @ v                                              # (B, H, T_q, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T_q, C)
        y = self.proj(y)
        return y, new_cache


class BlockKV(nn.Module):
    """支持 KV Cache 的 Block"""
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttentionKV(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)
    
    def forward(self, x, kv_cache=None):
        attn_out, new_cache = self.attn(self.ln1(x), kv_cache)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x, new_cache


class GPTKV(nn.Module):
    """支持 KV Cache 的 GPT"""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_embed = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([BlockKV(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight
    
    def forward(self, idx, kv_caches=None):
        B, T = idx.shape
        # 计算位置 offset (基于 cache 长度)
        if kv_caches is not None and kv_caches[0] is not None:
            past_len = kv_caches[0][0].size(2)
        else:
            past_len = 0
        pos = torch.arange(past_len, past_len + T, device=idx.device)
        
        x = self.token_embed(idx) + self.pos_embed(pos)
        
        new_caches = []
        for i, block in enumerate(self.blocks):
            cache_i = kv_caches[i] if kv_caches is not None else None
            x, new_cache = block(x, cache_i)
            new_caches.append(new_cache)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, new_caches


# 把训练好的权重迁移过来 (结构兼容)
model_kv = GPTKV(cfg).to(device)
sd = model.state_dict()
sd_kv = model_kv.state_dict()
# 名称完全一致 (我们设计成可以直接 load)
model_kv.load_state_dict(sd, strict=True)
print(f"权重迁移成功: {sum(p.numel() for p in model_kv.parameters()):,} 参数")


In [ ]:
@torch.no_grad()
def generate_kv(model, idx, max_new_tokens, temperature=1.0):
    """使用 KV Cache 的生成"""
    model.eval()
    
    # Prefill: 第一次输入完整 prompt，建立 cache
    idx_cond = idx if idx.size(1) <= cfg.block_size else idx[:, -cfg.block_size:]
    logits, kv_caches = model(idx_cond)
    
    for _ in range(max_new_tokens):
        # 只用最后一个 logits 采样
        logits_last = logits[:, -1, :] / temperature
        probs = F.softmax(logits_last, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
        
        # 用 KV Cache 增量前向: 只输入新 token
        # (注意要处理 block_size 截断，简化版假设不超过)
        if idx.size(1) > cfg.block_size:
            # cache 满了, 重置 (简化处理; 实际要做 sliding window)
            logits, kv_caches = model(idx[:, -cfg.block_size:])
        else:
            logits, kv_caches = model(idx_next, kv_caches=kv_caches)
    
    return idx


# Benchmark: 朴素 vs KV Cache
prompt = "ROMEO:"
n_gen = 200

torch.manual_seed(42)
context = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
t0 = time.time()
_ = generate_naive(model, context, max_new_tokens=n_gen, temperature=1.0)
naive_time = time.time() - t0

torch.manual_seed(42)
context = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
t0 = time.time()
_ = generate_kv(model_kv, context, max_new_tokens=n_gen, temperature=1.0)
kv_time = time.time() - t0

print("=" * 60)
print("KV Cache Benchmark")
print("=" * 60)
print(f"  朴素生成 ({n_gen} tokens):    {naive_time:.3f}s  ({n_gen/naive_time:.0f} tok/s)")
print(f"  KV Cache 生成 ({n_gen} tokens): {kv_time:.3f}s  ({n_gen/kv_time:.0f} tok/s)")
print(f"  加速比: {naive_time/kv_time:.2f}×")
print(f"\n💡 加速比在 block_size 越大、生成越长时越显著")
print(f"💡 GPT-2 1024 上下文生成 1024 token: KV Cache 提速 ~50×")


---
## 9. 采样策略对比 <a id='9'></a>

### 9.1 五种主流策略

| 策略 | 公式/做法 | 特点 |
|---|---|---|
| **Greedy** | `argmax(logits)` | 确定性、易陷重复、无创意 |
| **Temperature** | `softmax(logits / T)` | T<1 锐化 (更确定); T>1 平滑 (更随机) |
| **Top-k** | 取 top-k 后归一化采样 | 截断长尾，避免低概率噪声 |
| **Top-p (nucleus)** | 累积概率到 p 的最小集 | 自适应大小，常优于 top-k |
| **Beam Search** | 维护 b 个候选，全局最优 | 翻译/摘要常用，开放生成易乏味 |

### 9.2 Temperature 直觉

```
原始 logits: [2.0, 1.0, 0.5]
T = 1.0 → softmax: [0.66, 0.24, 0.10]
T = 0.5 → 等价于 logits ×2 → 更尖锐: [0.84, 0.11, 0.05]
T = 2.0 → 等价于 logits /2 → 更平滑: [0.46, 0.31, 0.23]
T → 0   → greedy
T → ∞   → uniform
```

### 9.3 Top-k vs Top-p

- **Top-k=50**: 每步固定取 50 个候选 — 简单但僵化
- **Top-p=0.9**: 取累积概率到 0.9 的最小集 — 高置信度时只取 1-2 个，低置信度时取很多个 (自适应)

主流 LLM API (GPT-4, Claude) 通常默认 `temperature=1.0, top_p=0.9` 左右。


In [ ]:
@torch.no_grad()
def sample_with_strategy(model, idx, max_new_tokens, strategy='multinomial', 
                          temperature=1.0, top_k=None, top_p=None):
    """统一的采样接口"""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx if idx.size(1) <= cfg.block_size else idx[:, -cfg.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        
        if strategy == 'greedy':
            idx_next = logits.argmax(dim=-1, keepdim=True)
        else:
            # 应用 top-k
            if top_k is not None and top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            # 应用 top-p (nucleus)
            if top_p is not None and 0 < top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum_probs = F.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
                # 移除累积概率 > top_p 的位置 (保留至少一个)
                mask = cum_probs > top_p
                mask[:, 1:] = mask[:, :-1].clone()
                mask[:, 0] = False
                # 在原顺序上 mask
                indices_to_remove = sorted_idx.gather(-1, mask.nonzero(as_tuple=True)[1].view(1, -1)) \
                    if mask.any() else None
                # 简化实现: 用 sorted 顺序 mask 后 scatter 回去
                sorted_logits[mask] = float('-inf')
                logits = torch.zeros_like(logits).scatter_(-1, sorted_idx, sorted_logits)
            
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


# 对比五种策略 (相同 prompt, 相同种子)
prompt = "ROMEO:\nWhere"
n_gen = 150
strategies = [
    ('greedy',                  dict(strategy='greedy')),
    ('T=1.0 (multinomial)',     dict(strategy='multinomial', temperature=1.0)),
    ('T=0.5 (sharper)',         dict(strategy='multinomial', temperature=0.5)),
    ('T=1.0 + top_k=10',        dict(strategy='multinomial', temperature=1.0, top_k=10)),
    ('T=1.0 + top_p=0.9',       dict(strategy='multinomial', temperature=1.0, top_p=0.9)),
]

print(f"Prompt: {prompt!r}\n")
for name, kwargs in strategies:
    torch.manual_seed(42)
    ctx = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    out = sample_with_strategy(model, ctx, n_gen, **kwargs)
    text = decode(out[0].tolist())
    # 只显示新生成的部分
    new_text = text[len(prompt):]
    print(f"--- {name} ---")
    print(repr(new_text[:120]))
    print()


In [ ]:
# 可视化: 不同 temperature 下的 next-token 概率分布
prompt = "ROMEO:"
ctx = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
with torch.no_grad():
    logits, _ = model(ctx)
last_logits = logits[0, -1, :].cpu()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, T in zip(axes, [0.3, 0.7, 1.0, 1.5]):
    probs = F.softmax(last_logits / T, dim=-1).numpy()
    top_indices = probs.argsort()[-10:][::-1]
    chars = [itos[i] for i in top_indices]
    chars = [c if c.strip() else f'\\x{ord(c):02x}' for c in chars]
    ax.bar(range(len(top_indices)), probs[top_indices], color='#1976D2')
    ax.set_xticks(range(len(top_indices)))
    ax.set_xticklabels(chars, rotation=45)
    ax.set_title(f'T = {T}')
    ax.set_ylabel('P(next char)')
    ax.set_ylim(0, 1.0)

plt.suptitle(f'Top-10 next-token probs after {prompt!r}', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 T 越小 → 分布越尖 (greedy 极限); T 越大 → 越平 (uniform 极限)")
print("💡 实际场景: 创意写作 T=1.0+top_p=0.9; 代码/事实问答 T=0.0~0.3")


---
## 10. 总结 & 思考题 <a id='10'></a>

### 今日核心知识点

1. **GPT 架构**: token+pos embed → N×Block (Pre-LN, MHSA+FFN) → LN → lm_head (与 embed tied)
2. **Causal Mask**: 下三角，softmax 前赋 -inf；位置 t 只能看 ≤ t
3. **KV Cache**: 缓存历史 K/V，把每步前向从 O(t²) 降到 O(t)，代价是 O(n·d·L) 显存
4. **采样策略**: greedy / temperature / top-k / top-p / beam，对应不同任务和创造性需求

### 面试高频问题

1. **Causal mask 为什么在 softmax 之前？** 之后用 0 会让 softmax 之前的分数被均分一部分，破坏归一化
2. **KV Cache 为什么不缓存 Q？** Q 每步都是新 token 的，没有历史可缓存
3. **KV Cache 显存怎么算？** `2 × n_layer × n_embd × seq_len × bytes`，是 LLM 推理的主要显存压力
4. **MQA / GQA 是什么？** 多个 Q 头共享同一组 K/V → 大幅减小 KV Cache (LLaMA-2-70B 用 GQA-8)，W3D4 详讲
5. **Pre-LN 为什么比 Post-LN 易训练？** 残差路径上没有 LN，梯度可以无衰减传到底层
6. **Weight tying 为什么有效？** 嵌入和输出都在同一个 V×C 子空间，共享反映了「相似 token 应该有相似输出概率」
7. **为什么 GPT 选 decoder-only 而不是 encoder-decoder？** 训练统一 (next-token)、scaling 友好、任务可统一为续写

### 与项目/简历的关联

- **本周的 Diffusion 项目** (D5/D6): 文生图的 prompt encoder 一般是 CLIP text encoder (transformer encoder)；可以对比 encoder-only / decoder-only 的设计哲学
- **博士论文 Ontology Prompting**: 你在用 GPT-4 做推理，这次手撕一遍能让你回答「prompt engineering 为什么 work」「ICL 为什么会涌现」
- **后续 W6 Agent 项目**: 微调 Qwen2-7B 是同样的 decoder-only 架构，只是规模放大 100×

### 明日预习: Diffusion 原理 ★

- DDPM 前向 (加噪) / 反向 (去噪) 直觉
- DDIM 加速采样
- Classifier-Free Guidance (CFG)
- ControlNet 架构 — 这是合成数据 Pipeline 的核心


In [ ]:
print("=" * 60)
print("W2 Day4 完成! 🎉")
print("=" * 60)
print(f"""
今日成果:
  ✅ 手写 Causal Self-Attention (合并 QKV + 多头分裂 + 因果 mask)
  ✅ 实现 Pre-LN Transformer Block (LN→Attn+→LN→FFN+)
  ✅ 完整 GPT 模型 ({model.num_params()/1e6:.2f}M params, weight tying)
  ✅ Tiny Shakespeare 字符级训练 ({max_iters} iters)
  ✅ KV Cache 实现 + benchmark ({naive_time/kv_time:.1f}× 加速)
  ✅ 五种采样策略对比 (greedy / T / top-k / top-p)
  ✅ Temperature 概率分布可视化

GitHub 上传清单 ("nanoGPT (GitHub)" 产出物):
  □ model.py: GPTConfig + CausalSelfAttention + Block + GPT
  □ model_kv.py: 带 KV Cache 的版本
  □ train.py: 训练循环 + estimate_loss
  □ sample.py: 五种采样策略
  □ data/prepare.py: tokenizer + 切分
  □ README.md: 架构图 + 训练曲线 + KV cache benchmark + 生成样本
  □ requirements.txt
""")